# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shrishagk/My_flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from getpass import getpass
import duckdb

HF_TOKEN = getpass("Enter your Hugging Face token: ")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

In [2]:

REL = "hf://datasets/FlyRank/internship-warehouse"

PERF_JAN = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2025-01/*.parquet'"
    f")"
)

print(con.sql(f"SELECT * FROM {PERF_JAN} LIMIT 5").df())

  report_date           client_hash_id           content_hash_id  \
0  2025-01-27  client_9958f0a7ae1df715  content_3b70a18ea133b2bb   
1  2025-01-27  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2   
2  2025-01-27  client_9958f0a7ae1df715  content_b4462a1b90640058   
3  2025-01-27  client_9958f0a7ae1df715  content_c899aef92518c714   
4  2025-01-27  client_9958f0a7ae1df715  content_c7c1d2e68d9d0964   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True            True                True               False   
1            True            True                True               False   
2            True            True                True               False   
3            True            True                True               False   
4            True            True                True               False   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               30           0               115  ...     

In [3]:

jan_sample = con.sql(f'SELECT * FROM {PERF_JAN}').df()

print(jan_sample.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I use a Random Forest Regressor because the lane is a ranking problem with mixed performance signals and potentially nonlinear relationships. The model is used to produce a continuous priority score that can be ranked, rather than as an automatic refresh decision. A tree-based model also gives a simple feature-importance view for error analysis.

In [4]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

print(model)

RandomForestRegressor(max_depth=6, min_samples_leaf=5, n_estimators=300,
                      n_jobs=-1, random_state=42)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I use a time-aware split because the goal is to rank content for future review. Training uses earlier observations and evaluation uses later observations, so information from the evaluation period does not enter model fitting. I avoid a random split because the same content item appears on multiple dates, which could make a random split overly optimistic.

In [6]:
# Start from the January data already loaded for ML-06/ML-07.
import pandas as pd
data = jan_sample.copy()
data["report_date"] = pd.to_datetime(data["report_date"])

print("Dates:")
print(sorted(data["report_date"].unique()))

train_end = pd.Timestamp("2025-01-29")

train = data[data["report_date"] <= train_end].copy()
test = data[data["report_date"] > train_end].copy()

print("\nTrain rows:", len(train))
print("Test rows:", len(test))

print("\nTrain dates:")
print(train["report_date"].value_counts().sort_index())

print("\nTest dates:")
print(test["report_date"].value_counts().sort_index())

Dates:
[Timestamp('2025-01-27 00:00:00'), Timestamp('2025-01-28 00:00:00'), Timestamp('2025-01-29 00:00:00'), Timestamp('2025-01-30 00:00:00'), Timestamp('2025-01-31 00:00:00')]

Train rows: 882
Test rows: 415

Train dates:
report_date
2025-01-27    303
2025-01-28    317
2025-01-29    262
Name: count, dtype: int64

Test dates:
report_date
2025-01-30    194
2025-01-31    221
Name: count, dtype: int64


In [7]:
# Sort by content and time.
data = data.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).copy()

# Next day's clicks for the same content item.
data["next_day_clicks"] = (
    data.groupby(
        ["client_hash_id", "content_hash_id"]
    )["gsc_clicks"]
    .shift(-1)
)

print(data[
    [
        "report_date",
        "content_hash_id",
        "gsc_clicks",
        "next_day_clicks"
    ]
].head(10))

     report_date           content_hash_id  gsc_clicks  next_day_clicks
79    2025-01-27  content_0217be03126aa7a5           0              0.0
381   2025-01-28  content_0217be03126aa7a5           0              0.0
697   2025-01-29  content_0217be03126aa7a5           0              0.0
957   2025-01-30  content_0217be03126aa7a5           0              0.0
1154  2025-01-31  content_0217be03126aa7a5           0              NaN
63    2025-01-27  content_0642dc7f62d4f780           1              1.0
366   2025-01-28  content_0642dc7f62d4f780           1              0.0
681   2025-01-29  content_0642dc7f62d4f780           0              0.0
941   2025-01-30  content_0642dc7f62d4f780           0              2.0
1138  2025-01-31  content_0642dc7f62d4f780           2              NaN


In [10]:
model_data = data.dropna(subset=["next_day_clicks"]).copy()

print("Rows with next-day target:", len(model_data))

Rows with next-day target: 821


In [11]:
import numpy as np
model_data["ctr"] = np.where(
    model_data["gsc_impressions"] > 0,
    model_data["gsc_clicks"] / model_data["gsc_impressions"],
    np.nan
)

model_data["total_sessions"] = (
    model_data["sessions_organic"].fillna(0)
    + model_data["sessions_direct"].fillna(0)
    + model_data["sessions_referral"].fillna(0)
    + model_data["sessions_social"].fillna(0)
    + model_data["sessions_paid"].fillna(0)
    + model_data["sessions_ai"].fillna(0)
)

model_data["organic_share"] = np.where(
    model_data["total_sessions"] > 0,
    model_data["sessions_organic"]
    / model_data["total_sessions"],
    np.nan
)

features = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events",
]

X = model_data[features].copy()
y = model_data["next_day_clicks"].copy()

# Tree models don't accept NaN.
X = X.fillna(0)

print("Features:", features)
print("X shape:", X.shape)
print("y shape:", y.shape)

Features: ['gsc_impressions', 'gsc_clicks', 'ctr', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_engaged_sessions', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events']
X shape: (821, 14)
y shape: (821,)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

I compare the model and the rule baseline on the same later-period observations and the same ranking metric. The evaluation uses next-day clicks as the observed proxy for review value because the dataset does not contain human review labels. NDCG@10 measures whether higher-value items are placed near the top of the ranked queue.

In [12]:
# Time-aware model split.
train_mask = model_data["report_date"] <= pd.Timestamp("2025-01-29")
test_mask = model_data["report_date"] > pd.Timestamp("2025-01-29")

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

print("Training rows:", len(X_train))
print("Evaluation rows:", len(X_test))

model.fit(X_train, y_train)

model_pred = model.predict(X_test)

print("Model trained.")

Training rows: 641
Evaluation rows: 180
Model trained.


In [13]:
eval_data = model_data.loc[test_mask].copy()

# A simple baseline using observed current-day signals.
eval_data["baseline_score"] = 0

ctr_threshold = model_data.loc[
    train_mask, "ctr"
].quantile(0.25)

position_threshold = model_data.loc[
    train_mask, "gsc_avg_position"
].quantile(0.75)

eval_data.loc[
    eval_data["ctr"] <= ctr_threshold,
    "baseline_score"
] += 2

eval_data.loc[
    eval_data["gsc_avg_position"] >= position_threshold,
    "baseline_score"
] += 2

print(eval_data[
    [
        "content_hash_id",
        "baseline_score",
        "next_day_clicks"
    ]
].head())

              content_hash_id  baseline_score  next_day_clicks
957  content_0217be03126aa7a5               2              0.0
941  content_0642dc7f62d4f780               2              2.0
965  content_0672d8db776419c0               2              0.0
931  content_0ca502c18c4fd41e               2              0.0
982  content_0d308caf94a3ed16               2              0.0


In [14]:
from sklearn.metrics import ndcg_score

baseline_ndcg10 = ndcg_score(
    [eval_data["next_day_clicks"].values],
    [eval_data["baseline_score"].values],
    k=10
)

model_ndcg10 = ndcg_score(
    [eval_data["next_day_clicks"].values],
    [model_pred],
    k=10
)

comparison = pd.DataFrame({
    "method": [
        "Rule baseline",
        "Random Forest"
    ],
    "NDCG@10": [
        baseline_ndcg10,
        model_ndcg10
    ]
})

display(comparison)

,method,NDCG@10
0,Rule baseline,0.000000
1,Random Forest,0.689987


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [15]:
eval_results = eval_data[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "next_day_clicks",
        "baseline_score"
    ]
].copy()

eval_results["model_prediction"] = model_pred

eval_results["absolute_error"] = (
    eval_results["next_day_clicks"]
    - eval_results["model_prediction"]
).abs()

display(
    eval_results.sort_values(
        "absolute_error",
        ascending=False
    ).head(10)
)

,client_hash_id,content_hash_id,report_date,next_day_clicks,baseline_score,model_prediction,absolute_error
1019,client_9958f0a7ae1df715,content_f94fe855380e150f,2025-01-30,5.0,0,2.115509,2.884491
953,client_9958f0a7ae1df715,content_37b3bafd5f88fdd1,2025-01-30,4.0,0,2.021440,1.978560
916,client_9958f0a7ae1df715,content_132cfd61ee6071be,2025-01-30,2.0,2,0.024318,1.975682
941,client_9958f0a7ae1df715,content_0642dc7f62d4f780,2025-01-30,2.0,2,0.110718,1.889282
988,client_9958f0a7ae1df715,content_4bb2b9cacd16054d,2025-01-30,2.0,0,0.194029,1.805971
954,client_9958f0a7ae1df715,content_de7b08874af74c00,2025-01-30,0.0,2,1.478184,1.478184
924,client_9958f0a7ae1df715,content_d02be57d816cf3d7,2025-01-30,0.0,0,1.165202,1.165202
920,client_9958f0a7ae1df715,content_4b90d8f71a8d9c59,2025-01-30,1.0,2,0.099441,0.900559
911,client_9958f0a7ae1df715,content_3921fa1f7890fd63,2025-01-30,1.0,2,0.157350,0.842650
932,client_9958f0a7ae1df715,content_67c0464f614c0606,2025-01-30,1.0,0,0.278477,0.721523


In [16]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance)

,feature,importance
0,gsc_impressions,0.763895
3,gsc_avg_position,0.210836
2,ctr,0.023575
1,gsc_clicks,0.001694
4,ga4_pageviews,0.000000
5,ga4_sessions,0.000000
6,ga4_engaged_sessions,0.000000
7,sessions_organic,0.000000
8,sessions_direct,0.000000
9,sessions_referral,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.